# 02 · Labels and features

Builds the modelling table with **leakage-safe** construction:

| Step | Rule |
|---|---|
| Bucket | `(ticker, weekday, slot)` → one observation per week |
| Split | chronological by **week**: 60% train / 20% val / 20% test |
| Label | per-ticker terciles of effective spread → easy / medium / hard; **thresholds from train only** |
| Features | only data from *earlier weeks* of the same bucket: last week's value and the trailing 4-week mean (RQ1) |
| Scaling | per-ticker robust z-score using that ticker's **train** statistics, so "hard" and feature values are relative to each stock's own normal |

Feature families (for RQ2 ablation): `spread_liquidity`, `price_deviation` (vs XLK/XLF), `oscillation`
(range, reversals, volume volatility, isotonic monotonicity error), plus `busyness` and `calendar`.

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from htt import config as C

import json
from htt import features
raw = pd.read_parquet(C.DATA_PROC / "raw_buckets.parquet")
sector_of = {s: k for k, v in C.SECTORS.items() for s in v["symbols"]}
bench_of  = {s: v["bench"] for k, v in C.SECTORS.items() for s in v["symbols"]}
ds, fam = features.build_dataset(raw, sector_of, bench_of)
print(ds.shape)
features.assert_no_leakage(ds)

In [ ]:
print({k: len(v) for k, v in fam.items()})
ds.groupby(["sector", "split"]).size().unstack()

## Label sanity checks

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ds.groupby("slot").y.apply(lambda s: (s == 2).mean()).plot(kind="bar", ax=ax[0], title="P(hard) by slot")
ds.groupby("symbol").y.apply(lambda s: (s == 2).mean()).sort_values().plot(kind="bar", ax=ax[1], title="P(hard) by ticker (train+val+test)")
plt.tight_layout()
print(ds.groupby("split").y.value_counts(normalize=True).unstack().round(3))

Terciles are set on **train**, so val/test class shares drift away from 1/3 if spreads trend. That regime
shift is real and worth a sentence in the report.

## Feature ↔ label relationship (quick look)

In [ ]:
tr = ds[ds.split == "train"]
corr = tr[fam["all"]].corrwith(tr.y, method="spearman").sort_values()
pd.concat([corr.head(8), corr.tail(8)]).plot(kind="barh", figsize=(7, 5), title="Spearman corr with class (train)")

In [ ]:
ds.to_parquet(C.DATA_PROC / "dataset.parquet")
json.dump(fam, open(C.DATA_PROC / "feature_families.json", "w"), indent=1)
print("saved", C.DATA_PROC)